# xbrl-frontier Phase 2: student QLoRA training on a Kaggle T4 (owner-run, resumable)

One run per session. All settings come from `docs/config/thresholds.yaml` in the attached repo snapshot. Set only the three run variables in the next code cell.

**Settings:** Accelerator = GPU T4 (one GPU is used), Internet = on.
**Attach datasets:** `xbrl-frontier-src` (the repo `git archive` tarball) and `xbrl-frontier-corpus` (`train.jsonl` and `dev.jsonl` from `data/corpus/`, checked against `data/corpus_report.json`). When resuming, also attach the previous session's output as a dataset and set `RESUME` to its path.
**After the session:** save the notebook output as a new Kaggle dataset version (it holds `checkpoints/`, `curve/`, `run.json` and, when finished, `final_adapter/`).

In [ ]:
MODEL = 'Qwen/Qwen3-1.7B'   # or 'Qwen/Qwen3-0.6B'
SEED = 1                    # one of pins.seeds
CURVE = True                # True only for the primary student, seed 1
VALIDATION = False          # True only for the curve-validation run (1.7B, seed 1, CURVE = False)
RESUME = None               # e.g. '/kaggle/input/xbrl-frontier-run-1p7b-s1' for a continuing session

In [ ]:
import glob, os, shutil, subprocess, sys
dst = '/kaggle/working/xbrl-frontier'
tarballs = glob.glob('/kaggle/input/xbrl-frontier-src/**/*.tar.gz', recursive=True)
trees = [os.path.dirname(os.path.dirname(os.path.dirname(p)))
         for p in glob.glob('/kaggle/input/xbrl-frontier-src/**/docs/config/thresholds.yaml', recursive=True)]
if tarballs:
    assert len(tarballs) == 1, tarballs
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['tar', '-xzf', tarballs[0], '-C', dst], check=True)
else:  # Kaggle auto-extracted the upload
    assert len(trees) == 1, trees
    shutil.copytree(trees[0], dst, dirs_exist_ok=True)
os.chdir(dst)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import yaml
pins = yaml.safe_load(open('docs/config/thresholds.yaml'))['pins']['software']
missing = [k for k in ('transformers', 'peft', 'bitsandbytes', 'accelerate') if not pins.get(k)]
assert not missing, f'pins.software empty for {missing}: owner approval needed first'
print(pins)

In [ ]:
!pip install -q "transformers=={pins['transformers']}" "peft=={pins['peft']}" "bitsandbytes=={pins['bitsandbytes']}" "accelerate=={pins['accelerate']}" "pyyaml==6.0.3" "pyarrow==25.0.1" "huggingface_hub"
import importlib.metadata as md
installed = {k: md.version(k) for k in ('transformers', 'peft', 'bitsandbytes', 'accelerate')}
bad = {k: (v, pins[k]) for k, v in installed.items() if v != pins[k]}
assert not bad, f'installed != pins.software: {bad}; restart the kernel and re-run this cell'
import torch, transformers, peft, bitsandbytes, accelerate
imported = {'transformers': transformers.__version__, 'peft': peft.__version__,
            'bitsandbytes': bitsandbytes.__version__, 'accelerate': accelerate.__version__}
bad = {k: (v, pins[k]) for k, v in imported.items() if v != pins[k]}
assert not bad, f'imported != pins.software: {bad}; restart the kernel (Run > Restart) and run all again'
print(imported, 'torch', torch.__version__, 'CUDA', torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
import glob, os
corpus = os.path.dirname(glob.glob('/kaggle/input/xbrl-frontier-corpus/**/train.jsonl', recursive=True)[0])
tag = MODEL.split('/')[-1] + f'-s{SEED}' + ('-validation' if VALIDATION else '')
out = f'/kaggle/working/run-{tag}'
cmd = f'python -m training.kaggle_train --model {MODEL} --seed {SEED} --corpus {corpus} --out {out}'
cmd += ' --curve' if CURVE else ''
cmd += ' --validation' if VALIDATION else ''
cmd += f' --resume {RESUME}' if RESUME else ''
print(cmd)
!{cmd}

In [ ]:
!ls -la {out} && cat {out}/run.json | head -60